### Load The Label Tree and define input / output path

In [1]:
import json
with open("../ressources/label_scheme.json") as f:
    label_tree = json.load(f)

with open("../ressources/meta.json") as f:
    meta = json.load(f)


In [2]:
from pathlib import Path
input_dir = Path("../data/Document_Échantillon_Initial/ronde_3/plain_html")
output_dir = Path("../data/Document_Échantillon_Initial/ronde_3/plain_html_arbre_balise")

html_files = list(input_dir.glob("*.html")) + list(input_dir.glob("*.htm"))
print(f"Found {len(html_files)} HTML/HTM files to process.\n")

Found 3 HTML/HTM files to process.



### Add the label tree in all file of the input_dir

In [3]:
from bs4 import BeautifulSoup
import chardet

# Process all HTML and HTM files in the input directory
html_files = list(input_dir.glob("*.html")) + list(input_dir.glob("*.htm"))
print(f"Found {len(html_files)} HTML/HTM files to process.\n")
for html_file in html_files:
    print(f"Processing: {html_file.name}")
    
    # Detect the encoding of the file
    with open(html_file, 'rb') as f:
        raw_data = f.read()
        detected = chardet.detect(raw_data)
        detected_encoding = detected['encoding']
        confidence = detected['confidence']
        print(f"  → Detected encoding: {detected_encoding} (confidence: {confidence:.2%})")
    
    # Read the original HTML content with detected encoding
    with open(html_file, 'r', encoding=detected_encoding) as f:
        content = f.read()
    
    # Parse the HTML
    soup = BeautifulSoup(content, 'html.parser')
    
    # Find the head tag
    head_tag = soup.find('head')
    
    if head_tag:
        # Create a comment node with the label tree
        from bs4 import Comment
        comment_text = f''' HTMLLabelizer
{{
  "labeltree": {json.dumps(label_tree, indent=4)},
  "meta": {json.dumps(meta, indent=4)}
}}
'''
        comment = Comment(comment_text)
        
        # Insert the comment before the head tag
        head_tag.insert_before(comment)
        
        # Save to output directory as UTF-8
        output_file = output_dir / html_file.name
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(str(soup))
        
        print(f"✓ Saved to: {output_file.name} (as UTF-8)\n")
    else:
        print(f"⚠ Warning: No <head> tag found in {html_file.name}\n")

print(f"\n✓ Processing complete! All files converted to UTF-8 and saved to {output_dir}")

Found 3 HTML/HTM files to process.

Processing: 2005QCCA437.html
  → Detected encoding: Windows-1252 (confidence: 73.00%)
✓ Saved to: 2005QCCA437.html (as UTF-8)

Processing: 2008CSC9.html
  → Detected encoding: Windows-1252 (confidence: 73.00%)
✓ Saved to: 2008CSC9.html (as UTF-8)

Processing: 2016NBOMB12.html
  → Detected encoding: Windows-1252 (confidence: 73.00%)
✓ Saved to: 2016NBOMB12.html (as UTF-8)


✓ Processing complete! All files converted to UTF-8 and saved to ..\data\Document_Échantillon_Initial\ronde_3\plain_html_arbre_balise
